# ATML PA4 — Task 3 (Colab Pro GPU Execution: Distillation & Speculative Decoding)

Select **L4** (recommended) or **T4** in `Runtime -> Change runtime type` and keep the same GPU tier for all Task 3 runs. All outputs save directly to Google Drive and auto-disconnect at the end.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/umairs5/atml-pa4.git'
DATA_REPO_URL = 'https://github.com/AbDu11aHHH/atml-pa4-datasets.git'
DRIVE_TASK3_DIR = '/content/drive/MyDrive/atml-pa4/results/task3'

In [ ]:
%cd /content
!if [ -d atml-pa4/.git ]; then git -C atml-pa4 pull --ff-only; else git clone {REPO_URL} atml-pa4; fi
%cd /content/atml-pa4
!if [ ! -d atml-pa4-datasets/.git ]; then git clone {DATA_REPO_URL} atml-pa4-datasets; fi
!cp configs/local_source_revisions.json atml-pa4-datasets/data/local_source_revisions.json
!nvidia-smi
!python -c "import torch; assert torch.cuda.is_available(), 'No CUDA GPU detected! Select L4 or T4 in Runtime -> Change runtime type.'"
!pip install -q -r requirements.txt

In [ ]:
# Cache prefix banks on Drive; use generate_prefix_banks_compat.py because setup_data.py passes generator= which transformers==4.57.1 rejects
!mkdir -p /content/drive/MyDrive/atml-pa4/cache atml-pa4-datasets/data/prefix_banks
!if [ -f /content/drive/MyDrive/atml-pa4/cache/prefix_banks_locked.jsonl.gz ]; then cp /content/drive/MyDrive/atml-pa4/cache/prefix_banks_locked.jsonl.gz atml-pa4-datasets/data/prefix_banks/; fi
!python atml-pa4-datasets/scripts/setup_data.py --public
!python scripts/generate_prefix_banks_compat.py --dataset-root ./atml-pa4-datasets
!if [ ! -f /content/drive/MyDrive/atml-pa4/cache/prefix_banks_locked.jsonl.gz ]; then cp atml-pa4-datasets/data/prefix_banks/prefix_banks_locked.jsonl.gz /content/drive/MyDrive/atml-pa4/cache/; fi
!python atml-pa4-datasets/scripts/check_setup.py
!python -m unittest tests/test_task3.py -v

In [ ]:
# Run all Task 3 phases (validation, off-policy, on-policy, prefix banks, spec benchmarks, standalone eval)
!python scripts/run_task3.py --dataset-root ./atml-pa4-datasets --output-dir {DRIVE_TASK3_DIR}

In [ ]:
# Analyze Task 3 results, compute paired bootstrap CIs, and download lightweight archive
!python scripts/analyze_task3.py --task3-dir {DRIVE_TASK3_DIR} --out-dir {DRIVE_TASK3_DIR}/analysis

import pandas as pd
for name in ('distillation_summary.csv', 'speculative_summary.csv', 'standalone_accuracy_summary.csv'):
    print('===', name, '===')
    display(pd.read_csv(f'{DRIVE_TASK3_DIR}/analysis/{name}'))

!cd {DRIVE_TASK3_DIR} && zip -rq /content/task3_results.zip env.json step1_validation.json merge_verification.json prefix_alignment_metrics.json distill/*.json spec_bench/ standalone/ analysis/
!cp /content/task3_results.zip {DRIVE_TASK3_DIR}/task3_results.zip
from google.colab import files
files.download('/content/task3_results.zip')

In [ ]:
# Flush Google Drive writes and disconnect/terminate runtime to save Colab compute units
import time
from google.colab import drive, runtime

time.sleep(15)
drive.flush_and_unmount()
runtime.unassign()